In [1]:
import mlflow
import pandas as pd
import numpy as np

from pathlib import Path

In [6]:
# Hubungkan notebook ke MLflow dan Cari eksperimen

from pathlib import Path
import mlflow

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

tracking_uri = f"sqlite:///{(project_dir / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("smart-personnel-workload")

print(f"Pakai DB: {tracking_uri}")

Pakai DB: sqlite:///c:/MLProjects/smart-personnel-mlops/mlflow.db


In [10]:
# Cari run Linear Regression

import pandas as pd

experiment = mlflow.get_experiment_by_name("smart-personnel-workload")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Bikin tabel versi modul (biar sama kayak foto kamu)
tabel = pd.DataFrame()
tabel["run_name"] = runs["tags.mlflow.runName"]
tabel["status"] = runs["status"]
tabel["MAE"] = runs["metrics.MAE"]
tabel["RMSE"] = runs["metrics.RMSE"]
tabel["R2"] = runs["metrics.R2"]

# HANYA ambil yang FINISHED biar yang FAILED ilang
tabel_bersih = tabel[tabel["status"] == "FINISHED"].sort_values("R2", ascending=False)

tabel_bersih

,run_name,status,MAE,RMSE,R2
0,linear_regression_v1,FINISHED,8.99111,11.347714,0.888424


In [13]:
# Ambil ID model Linear Regression

linear_run = runs[
    runs["tags.mlflow.runName"] == "linear_regression_v1"
].iloc[0]

run_id = linear_run["run_id"]

print("Run Name :", linear_run["tags.mlflow.runName"])
print("Run ID   :", run_id)

Run Name : linear_regression_v1
Run ID   : 7074b21b20ad41d5be015614d223ca90


In [14]:
# Load model dari MLflow

model_uri = f"runs:/{run_id}/model"

print("Model URI:")
print(model_uri)

Model URI:
runs:/7074b21b20ad41d5be015614d223ca90/model


In [15]:
loaded_model = mlflow.sklearn.load_model(model_uri)

print("Model berhasil dimuat dari MLflow!")
print("Tipe model:", type(loaded_model))

Model berhasil dimuat dari MLflow!
Tipe model: <class 'sklearn.pipeline.Pipeline'>


In [16]:
# Load data untuk pengujian

data_path = Path("../data/processed/employee_workload_feature_engineered.csv")

df = pd.read_csv(data_path)

print("Ukuran dataset:", df.shape)
display(df.head())

Ukuran dataset: (2000, 33)


,employee_id,department,division,position_level,employment_type,years_experience,age,working_days,working_hours,overtime_hours,...,customer_complaints,system_usage_hours,automation_level,remote_work_days,task_complexity,productivity_score,total_tasks_completed,document_completion_rate,pending_document_ratio,experience_category
0,EMP00001,Umum,Logistik,Supervisor,Tetap,2.3,31,20,158.6,55.0,...,1.0,94.5,0.252,0,0.839,0.663,234,1.0,0.0,Menengah
1,EMP00002,Pelayanan,Front Office,Senior Staff,Kontrak,6.0,24,23,193.5,1.5,...,3.0,99.1,0.133,4,0.782,0.557,286,1.0,0.0,Berpengalaman
2,EMP00003,Umum,Logistik,Staff,Kontrak,1.2,29,20,162.8,4.5,...,3.0,94.3,0.434,3,0.502,0.691,250,1.0,0.0,Pemula
3,EMP00004,Teknologi Informasi,Infrastruktur,Senior Staff,Tetap,6.4,25,21,158.2,1.7,...,1.0,116.3,0.543,6,0.417,0.796,218,1.0,0.0,Berpengalaman
4,EMP00005,Kepegawaian,Kesejahteraan,Junior Staff,Tetap,1.7,27,20,163.3,12.6,...,0.0,111.2,0.453,3,0.382,0.689,264,1.0,0.0,Pemula


In [19]:
print("\n".join(df.columns.tolist()))

employee_id
department
division
position_level
employment_type
years_experience
age
working_days
working_hours
overtime_hours
incoming_documents
processed_documents
pending_documents
service_requests
completed_requests
data_entry_tasks
verification_tasks
report_tasks
meeting_hours
training_hours
leave_days
average_processing_time
error_count
customer_complaints
system_usage_hours
automation_level
remote_work_days
task_complexity
productivity_score
total_tasks_completed
document_completion_rate
pending_document_ratio
experience_category


In [22]:
# Pisahkan fitur dan target

target_column = "total_tasks_completed"

X = df.drop(columns=[target_column, "employee_id"])
y = df[target_column]

print("Jumlah fitur:", X.shape[1])
print("Jumlah data :", X.shape[0])
print("Target      :", target_column)

Jumlah fitur: 31
Jumlah data : 2000
Target      : total_tasks_completed


In [23]:
# buat prediksi

predictions = loaded_model.predict(X)

print("Prediksi berhasil!")
print("Jumlah prediksi:", len(predictions))

display(
    pd.DataFrame({
        "Aktual": y.values[:10],
        "Prediksi": predictions[:10]
    })
)

Prediksi berhasil!
Jumlah prediksi: 2000


,Aktual,Prediksi
0,234,268.351868
1,286,283.385412
2,250,240.511883
3,218,244.743822
4,264,274.085902
5,210,207.454977
6,233,243.082659
7,200,217.896605
8,312,302.231451
9,339,326.365530


In [24]:
# Evaluasi model yang sudah di-load

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

mae = mean_absolute_error(y, predictions)
rmse = np.sqrt(mean_squared_error(y, predictions))
r2 = r2_score(y, predictions)

print("=== EVALUASI MODEL HASIL LOAD DARI MLFLOW ===")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

=== EVALUASI MODEL HASIL LOAD DARI MLFLOW ===
MAE  : 9.3679
RMSE : 11.7752
R²   : 0.8896


In [25]:
# Simpan salinan model lokal

import joblib

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

local_model_path = models_dir / "linear_regression_v1.pkl"

joblib.dump(loaded_model, local_model_path)

print("Model berhasil disimpan!")
print("Lokasi:", local_model_path.resolve())

Model berhasil disimpan!
Lokasi: C:\MLProjects\smart-personnel-mlops\models\linear_regression_v1.pkl


In [26]:
# test model lokal

local_model = joblib.load(local_model_path)

local_predictions = local_model.predict(X)

print("Model lokal berhasil dimuat!")
print("Jumlah prediksi:", len(local_predictions))

display(
    pd.DataFrame({
        "Aktual": y.values[:10],
        "Prediksi": local_predictions[:10]
    })
)

Model lokal berhasil dimuat!
Jumlah prediksi: 2000


,Aktual,Prediksi
0,234,268.351868
1,286,283.385412
2,250,240.511883
3,218,244.743822
4,264,274.085902
5,210,207.454977
6,233,243.082659
7,200,217.896605
8,312,302.231451
9,339,326.365530
